# 63 — Model Comparison
**Goal:** Compare LLM models on resume tasks — quality, cost, and speed.

Eight chapters of LLM engineering have assumed a model choice; this chapter makes the choice **evidence-based**. It lays out the evaluation dimensions that matter for resume tasks, prints a comparison matrix, and distills the results into per-task recommendations — the payoff of Ch. 55's tier system, now backed by numbers instead of vibes.

**Why it matters for resumes / ATS:** model choice is a real cost line once resumes scale. The cheapest model that hits your extraction accuracy target at your latency budget is the one to ship — and "cheapest" is only knowable if you benchmark on *your* tasks, because leaderboard rankings do not transfer to niche resume formats.

## 1. Evaluation Framework

The comparison has five dimensions, and all five are *task-specific*: a model can be excellent at creative generation and mediocre at structured extraction. **Accuracy** (does extraction match ground truth?), **Consistency** (same input, same output?), **Cost** (dollars per 1K calls), **Latency** (time to first token), and **Context window** (can it hold a full resume?).

**What the code does:** prints those five dimensions plus the four resume tasks worth benchmarking: skill extraction (structured output), bullet rewriting (generation), section classification (routing), and STAR generation (creative). The pairing matters — a task and a dimension only mean something together: e.g. consistency matters most for classification, latency matters most for interactive rewriting, context window matters for whole-resume analysis.

In [ ]:
print('''Model comparison dimensions for resume tasks:
1. Accuracy — Does extraction match ground truth?
2. Consistency — Same output for same input?
3. Cost — $ per 1K calls
4. Latency — Time to first token
5. Context window — Can it handle full resume?

Tasks to evaluate:
- Skill extraction (structured output)
- Bullet rewriting (generation)
- Section classification (routing)
- STAR generation (creative)''')

## 2. Building a Comparison Matrix

The matrix is the block's one-stop decision table: rows are models, columns are the five dimensions, and the printout formats them into a readable grid. **Caveat up front:** the numbers in this cell are hard-coded example data (the comment says so), not measured results — treat them as illustrative and re-benchmark on your own golden set before choosing.

**What the code does:**
- `comparison` dict — three models (`gpt-4o-mini`, `claude-3-haiku`, `mistral-small`) with `extraction_f1`, `rewrite_quality`, `cost_per_1k`, `latency_ms`, `context` values.
- The f-string header plus a loop prints one aligned row per model.

**Expected (verified by running):** the printed grid shows `gpt-4o-mini 0.94 F1, 4.2/5, $0.15, 800ms, 128k context`; `claude-3-haiku 0.93 F1, 4.0/5, $0.25, 600ms, 200k context`; `mistral-small 0.89 F1, 3.5/5, $0.05, 400ms, 32k context`. Read it as a trade-off surface, not a ranking: the cheapest model is also the slowest on quality, and the cheapest per call (mistral) has the smallest context window.

In [ ]:
# Model comparison data (example results)
comparison = {
    "gpt-4o-mini": {
        "extraction_f1": 0.94,
        "rewrite_quality": 4.2,
        "cost_per_1k": 0.15,
        "latency_ms": 800,
        "context": 128000,
    },
    "claude-3-haiku": {
        "extraction_f1": 0.93,
        "rewrite_quality": 4.0,
        "cost_per_1k": 0.25,
        "latency_ms": 600,
        "context": 200000,
    },
    "mistral-small": {
        "extraction_f1": 0.89,
        "rewrite_quality": 3.5,
        "cost_per_1k": 0.05,
        "latency_ms": 400,
        "context": 32000,
    },
}

print(f"""{"Model":25s} {"F1":6s} {"Quality":8s} {"Cost/1K":8s} {"Latency":8s}""")
print("-" * 55)
for model, data in comparison.items():
    print(f"{model:25s} {data['extraction_f1']:.2f}  {data['rewrite_quality']:.1f}/5   ${data['cost_per_1k']:.2f}    {data['latency_ms']}ms")

## 3. Recommendation by Task

A single winner is the wrong question — the right question is *which model for which task*. This cell prints a recommendation matrix that routes each of the five resume workloads to its best model and runner-up.

**What the code does:** prints the matrix, which pairs tasks with winners: skill extraction — GPT-4o-mini (runner-up Mistral Small); bullet rewrite — Claude 3 Haiku; STAR generation — GPT-4o-mini; classification — Mistral Small; career advice — GPT-4o-mini. It closes with the rule of thumb: `GPT-4o-mini is the best all-rounder for resume tasks`, with `Mistral Small` for batch classification (cheap, fast).

**Why it matters:** this matrix is exactly what Ch. 55's `pick_model()` encodes as tiers — the `cheap` tier serves classification, `balanced` serves extraction and rewriting, `best` serves generation — so the policy in code and the benchmark here agree by construction.

In [ ]:
print('''Recommendation matrix:
Task                Best Model          Runner-up
──────────────────────────────────────────────────
Skill extraction    GPT-4o-mini         Mistral Small
Bullet rewrite      Claude 3 Haiku      GPT-4o-mini
STAR generation     GPT-4o-mini         Llama 3 70B
Classification      Mistral Small       GPT-4o-mini
Career advice       GPT-4o-mini         Claude 3 Haiku

Rule of thumb: GPT-4o-mini is the best all-rounder for resume tasks.
Use Mistral Small for batch classification (cheap, fast).''')

## Summary: Benchmark models on your specific tasks. Don't assume — measure cost, quality, and latency.

**The cheapest model that meets your accuracy bar on *your* tasks is the right model — measure, don't assume.**

Five dimensions (accuracy, consistency, cost, latency, context) across four resume tasks, summarized into a per-task recommendation matrix that mirrors Ch. 55's tier policy: cheap models for classification, balanced for extraction and rewriting, flagship for generation. The example data is illustrative, but the method is the deliverable — a golden set, a matrix, and a routing rule.

This chapter closes the LLM engineering block; Ch. 64 (Precision and Recall for NLP) turns the qualitative "accuracy" dimension into proper evaluation metrics for the extraction tasks built here.